# Additional Analyses

Performance breakdown by diffusion method for detectors and ensembles.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, roc_curve
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 100

# ── Chemins ──────────────────────────────────────────────────────
BASE        = '/content/drive/MyDrive/Memoire_Deepfakes'
RESULTS_DIR = f'{BASE}/data/results'
FIGS_DIR    = f'{RESULTS_DIR}/figures'
os.makedirs(FIGS_DIR, exist_ok=True)

# ── Constantes ───────────────────────────────────────────────────
PROB_COLS   = ['P_Meso4', 'P_XceptionNet', 'P_UCF', 'P_F3Net']
MODEL_NAMES = ['Meso4',   'XceptionNet',   'UCF',   'F3Net']
LABEL_COL   = 'label'
METHOD_COL  = 'method'

COLORS_MODELS = {
    'Meso4'      : '#1f77b4',
    'XceptionNet': '#ff7f0e',
    'UCF'        : '#2ca02c',
    'F3Net'      : '#d62728',
}
COLORS_METHODS = {
    'MidJourney': '#9467bd',
    'ddim'      : '#8c564b',
    'DiT'       : '#e377c2',
    'SiT'       : '#17becf',
}
COLORS_ENSEMBLES = {
    'Scén. A — Vote maj.'   : '#9467bd',
    'Scén. B — Moy. pond.'  : '#8c564b',
    'Scén. C — Méta-Learn.' : '#e377c2',
}

# ── Chargement ───────────────────────────────────────────────────
train_df = pd.read_csv(f'{RESULTS_DIR}/train_probs.csv')
val_df   = pd.read_csv(f'{RESULTS_DIR}/val_probs.csv')

print('=' * 65)
print('CHARGEMENT val_probs.csv + train_probs.csv')
print('=' * 65)
print(f'  train_probs.csv : {len(train_df):,} lignes')
print(f'  val_probs.csv   : {len(val_df):,} lignes')
print(f'  Colonnes        : {list(val_df.columns)}')
print()

dist_label = val_df[LABEL_COL].value_counts().sort_index().to_dict()
print(f'  Labels Val : {dist_label}  (0=REAL, 1=FAKE)')

if METHOD_COL in val_df.columns:
    dist_method = val_df[val_df[LABEL_COL]==1][METHOD_COL].value_counts().to_dict()
    print(f'  Méthodes FAKE (Val) : {dist_method}')
else:
    raise KeyError('⚠️  Colonne "method" absente de val_probs.csv — vérifier NB06')

# ── Sous-ensembles ───────────────────────────────────────────────
real_df  = val_df[val_df[LABEL_COL] == 0].copy()
fake_df  = val_df[val_df[LABEL_COL] == 1].copy()
methods  = sorted(fake_df[METHOD_COL].unique().tolist())

print(f'\n  REAL : {len(real_df):,} images')
for m in methods:
    n = (fake_df[METHOD_COL] == m).sum()
    print(f'  FAKE / {m:<12s} : {n:,} images')

# ── Fonctions utilitaires ─────────────────────────────────────────
def compute_eer(y_true, y_scores):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    fnr = 1.0 - tpr
    idx = int(np.nanargmin(np.abs(fpr - fnr)))
    return float((fpr[idx] + fnr[idx]) / 2.0)

def compute_metrics(y_true, y_scores, threshold=0.5):
    y_pred = (np.asarray(y_scores) >= threshold).astype(int)
    return {
        'AUC'      : float(roc_auc_score(y_true, y_scores)),
        'Accuracy' : float(accuracy_score(y_true, y_pred)),
        'F1'       : float(f1_score(y_true, y_pred, zero_division=0)),
        'EER'      : compute_eer(y_true, y_scores),
    }

print()
print('✅ Setup terminé.')


## Ensemble Scores Calculation

In [ ]:
print('=' * 70)
print('CALCUL DES SCORES D\'ENSEMBLE POUR LE VALIDATION SET')
print('(Réutilisation des paramètres calibrés sur Train Set)')
print('=' * 70)

y_train = train_df[LABEL_COL].values
y_val   = val_df[LABEL_COL].values

# ────────────────────────────────────────────────────────────────
# SCÉNARIO A — Vote Majoritaire (moyenne simple)
# ────────────────────────────────────────────────────────────────
print('\n[1/3] Scénario A — Vote Majoritaire')
val_df['score_A'] = val_df[PROB_COLS].mean(axis=1)
print(f'  ✅ score_A calculé (moyenne simple des 4 probabilités)')

# ────────────────────────────────────────────────────────────────
# SCÉNARIO B — Moyenne Pondérée (poids ∝ accuracy train)
# ────────────────────────────────────────────────────────────────
print('\n[2/3] Scénario B — Moyenne Pondérée')
print('  Calibration des poids sur Train Set ...')

raw_weights = {}
for model, col in zip(MODEL_NAMES, PROB_COLS):
    pred_train = (train_df[col].values >= 0.5).astype(int)
    acc_train  = float(accuracy_score(y_train, pred_train))
    raw_weights[col] = acc_train
    print(f'    {model:<14s}  accuracy train = {acc_train:.4f}')

total_w      = sum(raw_weights.values())
norm_weights = {col: w / total_w for col, w in raw_weights.items()}

print('\n  Poids normalisés (somme = 1.0000) :')
for model, col in zip(MODEL_NAMES, PROB_COLS):
    print(f'    {model:<14s}  w = {norm_weights[col]:.4f}')

# Application au Val Set
val_df['score_B'] = sum(norm_weights[col] * val_df[col].values for col in PROB_COLS)
print(f'\n  ✅ score_B calculé (moyenne pondérée avec poids train)')

# ────────────────────────────────────────────────────────────────
# SCÉNARIO C — Méta-Learner (régression logistique)
# ────────────────────────────────────────────────────────────────
print('\n[3/3] Scénario C — Méta-Learner')
print('  Entraînement sur Train Set ...')

X_train = train_df[PROB_COLS].values
X_val   = val_df[PROB_COLS].values

clf = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
clf.fit(X_train, y_train)

print('\n  Coefficients appris (β) :')
for model, coef in zip(MODEL_NAMES, clf.coef_[0]):
    sign = '+' if coef >= 0 else ''
    print(f'    {model:<14s}  β = {sign}{coef:.4f}')
print(f'    Intercept       β0 = {clf.intercept_[0]:+.4f}')

# Application au Val Set
val_df['score_C'] = clf.predict_proba(X_val)[:, 1]
print(f'\n  ✅ score_C calculé (probabilité méta-learner)')

print()
print('=' * 70)
print('✅ Scores d\'ensemble calculés pour le Validation Set')
print('   Nouvelles colonnes : score_A, score_B, score_C')
print('=' * 70)


## Analysis A: Detectors by Method

In [ ]:
print('=' * 75)
print('ANALYSE A — BREAKDOWN PAR MÉTHODE DE DIFFUSION (DÉTECTEURS)')
print('Sous-dataset : images FAKE de la méthode + toutes les images REAL')
print('=' * 75)

results_A = {}

for method in methods:
    fake_subset = fake_df[fake_df[METHOD_COL] == method]
    subset_df   = pd.concat([real_df, fake_subset], ignore_index=True)
    y_subset    = subset_df[LABEL_COL].values

    n_real = (y_subset == 0).sum()
    n_fake = (y_subset == 1).sum()

    print(f'\n  ── {method} ──')
    print(f'     N = {len(subset_df):,}  (REAL={n_real:,}  FAKE={n_fake:,})')

    results_A[method] = {}
    for model, col in zip(MODEL_NAMES, PROB_COLS):
        scores = subset_df[col].values
        m = compute_metrics(y_subset, scores)
        results_A[method][model] = m
        print(f'     {model:<14s} AUC={m["AUC"]:.4f}  '
              f'Acc={m["Accuracy"]:.4f}  F1={m["F1"]:.4f}  EER={m["EER"]:.4f}')

print()
print('✅ Analyse A (détecteurs) terminée.')


## Analysis A2: Ensembles by Method

In [ ]:
print('=' * 75)
print('ANALYSE A2 — BREAKDOWN PAR MÉTHODE DE DIFFUSION (ENSEMBLES)')
print('Sous-dataset : images FAKE de la méthode + toutes les images REAL')
print('=' * 75)

results_A2 = {}
ENSEMBLE_COLS = ['score_A', 'score_B', 'score_C']
ENSEMBLE_NAMES = ['Scén. A — Vote maj.', 'Scén. B — Moy. pond.', 'Scén. C — Méta-Learn.']

for method in methods:
    # Fix: Create subsets from the updated val_df which contains the ensemble scores
    # The original real_df and fake_df were created before scores were added to val_df
    current_real_subset = val_df[val_df[LABEL_COL] == 0]
    current_fake_subset = val_df[(val_df[LABEL_COL] == 1) & (val_df[METHOD_COL] == method)]
    subset_df   = pd.concat([current_real_subset, current_fake_subset], ignore_index=True)
    y_subset    = subset_df[LABEL_COL].values

    n_real = (y_subset == 0).sum()
    n_fake = (y_subset == 1).sum()

    print(f'\n  ── {method} ──')
    print(f'     N = {len(subset_df):,}  (REAL={n_real:,}  FAKE={n_fake:,})')

    results_A2[method] = {}
    for ens_name, ens_col in zip(ENSEMBLE_NAMES, ENSEMBLE_COLS):
        scores = subset_df[ens_col].values
        m = compute_metrics(y_subset, scores)
        results_A2[method][ens_name] = m
        print(f'     {ens_name:<22s} AUC={m["AUC"]:.4f}  '
              f'Acc={m["Accuracy"]:.4f}  F1={m["F1"]:.4f}  EER={m["EER"]:.4f}')

print()
print('✅ Analyse A2 (ensembles) terminée.')

## Visualizations: Detectors

In [ ]:
# ── Extraction matrice AUC ────────────────────────────────────────
auc_matrix = pd.DataFrame({
    method: {model: results_A[method][model]['AUC'] for model in MODEL_NAMES}
    for method in methods
})

# ── Figure 1 : Heatmap AUC ────────────────────────────────────────
fig_A1, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    auc_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
    vmin=0.45, vmax=0.85, linewidths=0.5, cbar_kws={'label': 'AUC'},
    ax=ax
)
ax.set_xlabel('Méthode de Diffusion', fontsize=11)
ax.set_ylabel('Détecteur', fontsize=11)
ax.set_title(
    'AUC par Détecteur et Méthode de Diffusion (Val Set)\n'
    'Sous-datasets binaires : méthode FAKE + REAL',
    fontsize=11
)
plt.tight_layout()

path_A1 = f'{FIGS_DIR}/analysis_A_heatmap_auc.png'
fig_A1.savefig(path_A1, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure Analyse A1 sauvegardée : {path_A1}')

# ── Figure 2 : Barplot AUC ────────────────────────────────────────
fig_A2, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(methods))
width = 0.2

for i, model in enumerate(MODEL_NAMES):
    values = [results_A[m][model]['AUC'] for m in methods]
    ax.bar(x + i * width, values, width, label=model,
           color=COLORS_MODELS[model])

ax.set_xlabel('Méthode de Diffusion', fontsize=11)
ax.set_ylabel('AUC', fontsize=11)
ax.set_title('AUC par Détecteur et Méthode de Diffusion (Val Set)', fontsize=11)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(methods)
ax.legend(title='Détecteur', fontsize=9)
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Chance')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

path_A2 = f'{FIGS_DIR}/analysis_A_barplot_auc.png'
fig_A2.savefig(path_A2, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure Analyse A2 sauvegardée : {path_A2}')

# ── Figure 3 : Violin Plot Distribution P(FAKE) ──────────────────
plot_data = []
for method in methods:
    fake_subset = fake_df[fake_df[METHOD_COL] == method]
    for model, col in zip(MODEL_NAMES, PROB_COLS):
        for p in fake_subset[col].values:
            plot_data.append({'Méthode': method, 'Modèle': model, 'P_FAKE': p})

plot_df = pd.DataFrame(plot_data)

fig_A3, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(
    data=plot_df, x='Méthode', y='P_FAKE', hue='Modèle',
    palette=[COLORS_MODELS[m] for m in MODEL_NAMES],
    split=False, inner='quartile', ax=ax
)
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.set_ylabel('P(FAKE) — Probabilité prédite', fontsize=11)
ax.set_xlabel('Méthode de Diffusion', fontsize=11)
ax.set_title(
    'Distribution P(FAKE) par Détecteur et Méthode (Val Set — FAKE uniquement)',
    fontsize=11
)
ax.legend(title='Détecteur', fontsize=9, loc='upper right')
plt.tight_layout()

path_A3 = f'{FIGS_DIR}/analysis_A_violin_pfake.png'
fig_A3.savefig(path_A3, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure Analyse A3 sauvegardée : {path_A3}')


## Visualizations: Ensembles

In [ ]:
# ── Extraction matrice AUC (ensembles) ────────────────────────────
auc_matrix_ens = pd.DataFrame({
    method: {ens: results_A2[method][ens]['AUC'] for ens in ENSEMBLE_NAMES}
    for method in methods
})

# ── Figure A2-1 : Heatmap AUC (ensembles) ─────────────────────────
fig_A2_1, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    auc_matrix_ens, annot=True, fmt='.3f', cmap='RdYlGn',
    vmin=0.45, vmax=0.85, linewidths=0.5, cbar_kws={'label': 'AUC'},
    ax=ax
)
ax.set_xlabel('Méthode de Diffusion', fontsize=11)
ax.set_ylabel('Scénario d\'Ensemble', fontsize=11)
ax.set_title(
    'AUC par Ensemble et Méthode de Diffusion (Val Set)\n'
    'Sous-datasets binaires : méthode FAKE + REAL',
    fontsize=11
)
plt.tight_layout()

path_A2_1 = f'{FIGS_DIR}/analysis_A2_heatmap_auc_ensembles.png'
fig_A2_1.savefig(path_A2_1, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure Analyse A2-1 sauvegardée : {path_A2_1}')

# ── Figure A2-2 : Barplot AUC (ensembles) ─────────────────────────
fig_A2_2, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(methods))
width = 0.25

for i, ens_name in enumerate(ENSEMBLE_NAMES):
    values = [results_A2[m][ens_name]['AUC'] for m in methods]
    ax.bar(x + i * width, values, width, label=ens_name,
           color=COLORS_ENSEMBLES[ens_name])

ax.set_xlabel('Méthode de Diffusion', fontsize=11)
ax.set_ylabel('AUC', fontsize=11)
ax.set_title('AUC par Ensemble et Méthode de Diffusion (Val Set)', fontsize=11)
ax.set_xticks(x + width)
ax.set_xticklabels(methods)
ax.legend(title='Scénario', fontsize=9)
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Chance')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

path_A2_2 = f'{FIGS_DIR}/analysis_A2_barplot_auc_ensembles.png'
fig_A2_2.savefig(path_A2_2, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure Analyse A2-2 sauvegardée : {path_A2_2}')

# ── Figure A2-3 : Comparaison Détecteurs vs Ensembles ────────────
fig_A2_3, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, method in enumerate(methods):
    ax = axes[idx]

    # AUC des détecteurs individuels
    det_aucs = [results_A[method][model]['AUC'] for model in MODEL_NAMES]
    # AUC des ensembles
    ens_aucs = [results_A2[method][ens]['AUC'] for ens in ENSEMBLE_NAMES]

    x_pos = np.arange(len(MODEL_NAMES) + len(ENSEMBLE_NAMES))
    colors = [COLORS_MODELS[m] for m in MODEL_NAMES] + [COLORS_ENSEMBLES[e] for e in ENSEMBLE_NAMES]
    labels = MODEL_NAMES + ['A', 'B', 'C']  # Labels courts pour ensembles

    ax.bar(x_pos, det_aucs + ens_aucs, color=colors, alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, fontsize=9, rotation=45, ha='right')
    ax.set_ylabel('AUC', fontsize=10)
    ax.set_title(f'{method}', fontsize=11, fontweight='bold')
    ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0.4, 0.9)

plt.suptitle(
    'Comparaison Détecteurs vs Ensembles par Méthode de Diffusion\n'
    '(Val Set — A=Vote maj., B=Moy. pond., C=Méta-Learn.)',
    fontsize=12, y=0.995
)
plt.tight_layout()

path_A2_3 = f'{FIGS_DIR}/analysis_A2_comparison_det_vs_ens.png'
fig_A2_3.savefig(path_A2_3, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure Analyse A2-3 sauvegardée : {path_A2_3}')


## Analysis B: Error Correlation

In [ ]:
print('=' * 70)
print('ANALYSE B — CORRÉLATION DES ERREURS INTER-MODÈLES')
print('=' * 70)

errors_df = pd.DataFrame()

for model, col in zip(MODEL_NAMES, PROB_COLS):
    pred = (val_df[col].values >= 0.5).astype(int)
    errors_df[model] = (pred != y_val).astype(int)  # 1 = erreur

corr_matrix = errors_df.corr()

print('\nMatrice de corrélation des erreurs (1 = erreur) :')
print(corr_matrix.round(3))

# ── Distribution des accords ─────────────────────────────────────
n_errors = errors_df.sum(axis=1)  # nombre de modèles en erreur par image
dist = n_errors.value_counts().sort_index().to_dict()

print('\nDistribution du nombre de modèles en erreur par image :')
for k in range(5):
    n = dist.get(k, 0)
    pct = 100 * n / len(val_df)
    print(f'  {k} modèle(s) en erreur : {n:,} images ({pct:.2f}%)')

# ── Sauil d'erreur incompressible ────────────────────────────────
competent_models = ['XceptionNet', 'UCF', 'F3Net']
errors_competent = errors_df[competent_models]
all_wrong = (errors_competent.sum(axis=1) == 3).sum()
pct_incomp = 100 * all_wrong / len(val_df)

print(f'\n⚠️  Plafond d\'erreur incompressible :')
print(f'    Images ratées par les 3 modèles compétents : {all_wrong:,} ({pct_incomp:.2f}%)')
print(f'    → Ces images ne peuvent être récupérées par aucun ensemble')

print()
print('✅ Analyse B terminée.')


# CELLULE 9 — Visualisations Analyse B


In [ ]:
from matplotlib.lines import Line2D

fig_B, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Sous-figure 1 : Heatmap corrélation ──────────────────────────
ax = axes[0]
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, center=0, linewidths=0.5,
    cbar_kws={'label': 'Corrélation'}, ax=ax
)
ax.set_title(
    'Corrélation des Erreurs (Val Set)\n'
    '1 = erreur, 0 = correct',
    fontsize=11
)

# ── Sous-figure 2 : Scatter XceptionNet vs UCF ───────────────────
ax = axes[1]
real_mask = (val_df[LABEL_COL] == 0)
fake_mask = (val_df[LABEL_COL] == 1)

ax.scatter(
    val_df.loc[real_mask, 'P_XceptionNet'],
    val_df.loc[real_mask, 'P_UCF'],
    c='#2ca02c', s=10, alpha=0.3, edgecolors='none'
)
ax.scatter(
    val_df.loc[fake_mask, 'P_XceptionNet'],
    val_df.loc[fake_mask, 'P_UCF'],
    c='#d62728', s=10, alpha=0.3, edgecolors='none'
)

legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#2ca02c',
           markersize=8, label='REAL'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#d62728',
           markersize=8, label='FAKE'),
]
ax.legend(handles=legend_elements, fontsize=9)
ax.set_xlabel('P_XceptionNet', fontsize=10)
ax.set_ylabel('P_UCF', fontsize=10)
ax.set_title(
    'P(FAKE) XceptionNet vs UCF\n'
    '(Val Set — vert=REAL, rouge=FAKE)',
    fontsize=11
)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.2)

# ── Corrélation score brut XceptionNet vs UCF ─────────────────────
r_xc_ucf = np.corrcoef(
    val_df['P_XceptionNet'].values,
    val_df['P_UCF'].values
)[0,1]
ax.text(0.05, 0.95, f'r = {r_xc_ucf:.3f}',
        transform=ax.transAxes, fontsize=10,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.suptitle(
    'Analyse B — Complémentarité inter-modèles (Val Set)',
    fontsize=12, y=1.01
)
plt.tight_layout()

path_B = f'{FIGS_DIR}/analysis_B_inter_model.png'
fig_B.savefig(path_B, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure Analyse B sauvegardée : {path_B}')
